# Pre-process for tuning/training/evaluation workflow

- Subset 2014-2023 development data
    - 2014-2021 training (~78%)
    - 2022-2023 validation (~15%)
- Subset 2024 hold-out test data (~7%)
- Only use patients diagnosed w/ cancer

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.preprocess import transform_export_data_full
import pandas as pd

base_df = pd.read_parquet()

In [ ]:
col_types = {
    "meta": [
        "CASEID",
        "CPT",
    ],
    "demographics": [
        "SEX",
        "RACE",
        "HISPANIC",
        "AGE",
        "HEIGHT",
        "WEIGHT",
    ],
    "comorbidities": [
        "SMOKE",
        "DIABETES",
        "HXCOPD",
        "DYSPNEA",
        "HXCHF",
        "HYPERMED",
        "DIALYSIS",
        "RENAFAIL",
        "BLEEDDIS",
        "WTLOSS",
        "ASCITES",
        "STEROID",
        "DISCANCR",
        "WNDINF",
        "ASACLAS",
        "TRANSFUS",
    ],
    "blood": [
        "PRALBUM",
        "PRWBC",
        "PRHCT",
        "PRPLATE",
    ],
    "intra_op": [
        "SURGINDICD",
        "OPTIME",
        "OPERYR",
        "URGENCY",
        "ANESTHES",
        "SURGSPEC",
        "INOUT",
    ],
    "cpt_op": [
        ## Resection
        "PARTIALCPT",
        "SUBSIMPLECPT",
        "RADICALCPT",
        "MODIFIEDRADICALCPT",
        ## axillary
        "SNLBCPT",
        "ALNDCPT",
        ## implant-based
        "IMMEDIATECPT",
        "DELAYEDCPT",
        "TEINSERTIONCPT",
        "TEEXPANDERCPT",
        ## autologous
        "FREECPT",
        "LATCPT",
        "SINTRAMCPT",
        "BITRAMCPT",
        "SINTRAMSUPERCPT",
        ## adjunct + revision
        "AUGPROSIMPCPT",
        "MASTOCPT",
        "BREASTREDCPT",
        "FATGRAFTCPT",
        "REVRECBREASTCPT",
        "ADJTISTRANSCPT",
        "NPWTCPT",
        "OTHERRECONTECHCPT",
    ],
    "outcomes": [
        "SERIOUS",
        "ANY",
        "PNEUMO",
        "CARDIAC_COMP",
        "DVT",
        "SEPSIS",
        "SSI",
        "UTI",
        "RENAL",
        "UNPLNREOP",
        "MORT",
    ],
    "post_op_comps": [
        "TOTHLOS",
        "SUPINFEC",
        "WNDINFD",
        "ORGSPCSSI",
        "DEHIS",
        "REINTUB",
        "PULEMBOL",
        "FAILWEAN",
        "RENAINSF",
        "OPRENAFL",
        "CNSCVA",
        "CDARREST",
        "CDMI",
        "OTHBLEED",
        "OTHSYSEP",
        "OTHSESHOCK",
        "READ",
        "UNPLNREAD",
        "DISCHDEST",
    ],
    "misc": [
        "FNSTATUS2",
        "VENTILAT",
        "PRSEPIS",
        "PROCANATCPT",
        "NOLYMPH",
        "SURGTIMINGCPT",
    ],
}

In [ ]:
outcome_list = [
    "SERIOUS",
    "ANY",
    "PNEUMO",
    "CARDIAC_COMP",
    "VTE",
    "SEPSIS",
    "SSI",
    "UTI",
    "RENAL",
    "UNPLNREOP",
    "MORT",
]
selected_x = [
    # "BMI",
    "HEIGHT",
    "WEIGHT",
    "INOUT",
    "PARTIALCPT",
    "FREECPT",
    "SMOKE",
    "SUBSIMPLECPT",
    "ASACLAS",
    "TEINSERTIONCPT",
    "RACE",
    "DIABETES",
    "IMMEDIATECPT",
    "SURGSPEC",
    "BLEEDDIS",
    "PRALBUM",
    "HYPERMED",
    "PRWBC",
    "PRHCT",
    "HXCHF",
    "ALNDCPT",
    "SINTRAMCPT",
    "PRPLATE",
    "DISCANCR",
    "HXCOPD",
    "AGE",
    "BREASTREDCPT",
    "SNLBCPT",
    "URGENCY",
    "MODIFIEDRADICALCPT",
    "ADJTISTRANSCPT",
    "ANESTHES",
]

full_x = (
    col_types["demographics"]
    + col_types["comorbidities"]
    + col_types["blood"]
    + col_types["intra_op"]
    + col_types["cpt_op"]
)

run in parallel

In [ ]:
from joblib import Parallel, delayed

data_path = BASE_PATH / "data/processed"
pipeline_path = BASE_PATH / "data/pipelines"
imp_path = BASE_PATH / "data" / "raw" / "cleaned" / "NSQIP_mast_combined_cancer.parquet"


def _run(outcome_name):
    transform_export_data_full(
        df_path=imp_path,
        x_cols=selected_x,
        target_col_name=outcome_name,
        data_path=data_path,
        pipeline_path=pipeline_path,
    )


Parallel(n_jobs=-1)(delayed(_run)(outcome_name) for outcome_name in outcome_list)